In [28]:
import pandas as pd

adult_census = pd.read_csv("./dataset/adult-census.csv")

In [29]:
target_name = "class"
target = adult_census[target_name]
data = adult_census.drop(columns=[target_name, "education-num"])

In [30]:
from sklearn.compose import make_column_selector as selector

numerical_columns_selector = selector(dtype_exclude=object)
categorical_columns_selector = selector(dtype_include=object)
numerical_columns = numerical_columns_selector(data)
categorical_columns = categorical_columns_selector(data)

In [31]:
import time

from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

categorical_preprocessor = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1
)
preprocessor = make_column_transformer(
    (categorical_preprocessor, categorical_columns),
    remainder="passthrough",
)


model = make_pipeline(preprocessor, HistGradientBoostingClassifier(max_iter=500))

start = time.time()
cv_results = cross_validate(model, data, target)
elapsed_time = time.time() - start

scores = cv_results["test_score"]

print(
    "The mean cross-validation accuracy is: "
    f"{scores.mean():.3f} ± {scores.std():.3f} "
    f"with a fitting time of {elapsed_time:.3f} seconds"
)

The mean cross-validation accuracy is: 0.874 ± 0.003 with a fitting time of 2.599 seconds


In [32]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_preprocessing = StandardScaler()
categorical_preprocessor_onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

|  | Meaningful order | Non-meaningful order |
| -------- | ------- | -------------------- |
| Tree-based model  | OrdinalEncoder   | OrdinalEncoder with reasonable depth |
| Linear model | OrdinalEncoder with caution  | OneHotEncoder |

In [33]:
preprocessor = make_column_transformer(
    (categorical_preprocessor_onehot, categorical_columns),
    (numerical_preprocessing, numerical_columns),
    remainder="passthrough",
)


model = make_pipeline(preprocessor, HistGradientBoostingClassifier(max_iter=500))

start = time.time()
cv_results = cross_validate(model, data, target)
elapsed_time = time.time() - start

scores = cv_results["test_score"]

print(
    "The mean cross-validation accuracy is: "
    f"{scores.mean():.3f} ± {scores.std():.3f} "
    f"with a fitting time of {elapsed_time:.3f} seconds"
)

The mean cross-validation accuracy is: 0.873 ± 0.003 with a fitting time of 4.450 seconds
